# Exploratory data analysis

Both datasets, and the evidence behind the modelling choices. Run from the project root after `python -m src.data.download` and `python -m src.data.features`.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import pandas as pd
import matplotlib.pyplot as plt
from src import config
from src.data.features import load_ai4i, load_azure_raw, AI4I_MODES

## AI4I 2020: production cycles
10,000 cycles, 3.4% failures. Failure modes follow known physical rules, which is why derived features (temperature gap, power, strain) matter more than raw readings.

In [ ]:
ai4i = load_ai4i()
print(ai4i.shape)
display(ai4i[['machine_failure', *AI4I_MODES, 'RNF']].sum().rename('count').to_frame())
display(ai4i.groupby('type')['machine_failure'].agg(['count', 'mean']))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
ok, bad = ai4i[ai4i.machine_failure == 0], ai4i[ai4i.machine_failure == 1]
for ax, (x, y) in zip(axes, [('rpm', 'torque_nm'), ('rpm', 'temp_diff_k'), ('tool_wear_min', 'torque_nm')]):
    ax.scatter(ok[x], ok[y], s=3, alpha=0.2, color='#2a78d6', label='no failure')
    ax.scatter(bad[x], bad[y], s=10, color='#e34948', label='failure')
    ax.set_xlabel(x); ax.set_ylabel(y)
axes[0].legend()
axes[0].set_title('Power failures: extremes of speed x torque')
axes[1].set_title('Heat dissipation: low temp gap and low speed')
axes[2].set_title('Overstrain: high wear x torque')
plt.tight_layout()

## Azure predictive maintenance: fleet telemetry
100 machines, hourly readings for 2015, plus errors, replacements and failures.

In [ ]:
az = load_azure_raw()
for name, df in az.items():
    print(f'{name:10s} {df.shape}')
display(az['failures']['failure'].value_counts().to_frame('failures'))
display(az['machines']['model'].value_counts().to_frame('machines'))

In [ ]:
# Sensor behaviour in the days before a failure (machine with the most failures)
mid = az['failures']['machineID'].value_counts().index[0]
tel = az['telemetry'][az['telemetry'].machineID == mid].set_index('datetime')
fails = az['failures'][az['failures'].machineID == mid]
fig, axes = plt.subplots(4, 1, figsize=(15, 9), sharex=True)
for ax, s in zip(axes, ['volt', 'rotate', 'pressure', 'vibration']):
    ax.plot(tel[s].rolling(24).mean(), color='#2a78d6', lw=1.5)
    for t in fails['datetime']:
        ax.axvline(t, color='#e34948', lw=1, ls='--')
    ax.set_ylabel(s)
axes[0].set_title(f'Machine {mid}: 24h rolling sensor means, dashed lines = failures')
plt.tight_layout()

In [ ]:
# Label balance after feature engineering
feats = pd.read_parquet(config.DATA_PROCESSED / 'azure_features.parquet')
labels = [c for c in feats.columns if c.startswith('fail_')]
feats[labels].mean().rename('positive rate').to_frame()